<a href="https://colab.research.google.com/github/Priyanshu-Bais/AI-complaint-analyser-for-government-portal-using-NLP-/blob/main/Minor_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv("complaints_dataset_10k.csv")

print(df.shape)
print(df.head())
print(df.columns)

(10000, 3)
                                           complaint         category urgency
0  There are no lights on the Green Park bypass s...  roads_transport    high
1  Hello sir, garbage has been piling up at the S...       sanitation    high
2  Submitting this complaint to inform that the p...     water_supply  medium
3  Heavy waterlogging on the Rajiv Nagar underpas...  roads_transport    high
4  I wish to report that after last night's storm...      electricity    high
Index(['complaint', 'category', 'urgency'], dtype='object')


# Exploratory data anlysis

In [3]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nCategory distribution:")
print(df["category"].value_counts())

print("\nUrgency distribution:")
print(df["urgency"].value_counts())

Dataset shape: (10000, 3)

Missing values:
complaint    0
category     0
urgency      0
dtype: int64

Category distribution:
category
water_supply       1668
electricity        1667
healthcare         1667
roads_transport    1666
sanitation         1666
ration_aadhar      1666
Name: count, dtype: int64

Urgency distribution:
urgency
high      3334
low       3334
medium    3332
Name: count, dtype: int64


In [4]:
print("Duplicate complaints:", df["complaint"].duplicated().sum())

Duplicate complaints: 0


In [5]:
label_check = df.groupby("complaint")[["category", "urgency"]].nunique()

print("Complaints with different category labels:",
      (label_check["category"] > 1).sum())

print("Complaints with different urgency labels:",
      (label_check["urgency"] > 1).sum())

Complaints with different category labels: 0
Complaints with different urgency labels: 0


## Importing important libraries


In [6]:
import nltk
import re
import pandas as pd

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


# Applying the Pre-process function which does these work


*   converts to lowercase
*   cleaning ( remove Punctation)
*   tokenize
*   Remove stopword
*   Lemitization


In [7]:
def preprocess(text):

    if pd.isna(text):
        return ""

    text = str(text)
    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9]", " ", text)   # Remove punctuation

    tokens = word_tokenize(text)                # Tokenization

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]                                            # Remove stopwords

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]                                            # Lemmatization

    return " ".join(tokens)

In [10]:
df["Processed_Complaint"] = df["complaint"].apply(preprocess)
print(df[["complaint", "Processed_Complaint"]].head())

                                           complaint  \
0  There are no lights on the Green Park bypass s...   
1  Hello sir, garbage has been piling up at the S...   
2  Submitting this complaint to inform that the p...   
3  Heavy waterlogging on the Rajiv Nagar underpas...   
4  I wish to report that after last night's storm...   

                                 Processed_Complaint  
0  light green park bypass stretch two accident h...  
1  hello sir garbage piling sector 11 market four...  
2  submitting complaint inform piepline sector 11...  
3  heavy waterlogging rajiv nagar underpass trap ...  
4  wish report last night storm wire jawahar ward...  


# POS Tagging

In [11]:
nltk.download('averaged_perceptron_tagger_eng')

from nltk import pos_tag

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [12]:
def pos_tagging(text):
    tokens = word_tokenize(text)
    return pos_tag(tokens)

In [16]:
df["POS_Tags"] = df["Processed_Complaint"].apply(pos_tagging)

print(df[["Processed_Complaint", "POS_Tags"]].head())

                                 Processed_Complaint  \
0  light green park bypass stretch two accident h...   
1  hello sir garbage piling sector 11 market four...   
2  submitting complaint inform piepline sector 11...   
3  heavy waterlogging rajiv nagar underpass trap ...   
4  wish report last night storm wire jawahar ward...   

                                            POS_Tags  
0  [(light, NN), (green, JJ), (park, NN), (bypass...  
1  [(hello, NN), (sir, NN), (garbage, NN), (pilin...  
2  [(submitting, VBG), (complaint, NN), (inform, ...  
3  [(heavy, JJ), (waterlogging, VBG), (rajiv, JJ)...  
4  [(wish, JJ), (report, NN), (last, JJ), (night,...  


# Chunking the processed complaint to run them into NER

In [17]:
from nltk import ne_chunk

nltk.download('words')
nltk.download('maxent_ne_chunker_tab')

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.


True

# As of now we are using NLTK's provieded NER but latter at the time after deployment remember to use a NER which WIll identify the indians location more efffectively Like

* SpyCy

In [18]:
sample_tokens = word_tokenize(df["Processed_Complaint"].iloc[0])
sample_pos = pos_tag(sample_tokens)

ner_result = ne_chunk(sample_pos)

print(ner_result)

(S
  light/NN
  green/JJ
  park/NN
  bypass/NN
  stretch/NN
  two/CD
  accident/NN
  happened/VBD
  week/NN
  appreciate/NN
  help/NN)


# TF-IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df["Processed_Complaint"])

print("TF-IDF shape:", X.shape)

TF-IDF shape: (10000, 1798)


# Create the Category target

In [20]:
y_category = df["category"]

print(y_category.value_counts())

category
water_supply       1668
electricity        1667
healthcare         1667
roads_transport    1666
sanitation         1666
ration_aadhar      1666
Name: count, dtype: int64


## Train - test spliit

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_category,
    test_size=0.2,
    random_state=42,
    stratify=y_category
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (8000, 1798)
Testing data: (2000, 1798)


TF-IDF
   ↓
8,000 training complaints
*   ├── Naive Bayes
 *  ├── Logistic Regression
 *  └── Linear SVM
       *   ↓
 *    Compare results

# Training The models

In [22]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# Linear SVM
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

print("All three category models trained successfully.")

All three category models trained successfully.


# Evaluating the models

In [23]:
from sklearn.metrics import accuracy_score, classification_report

print("Naive Bayes Accuracy:")
print(accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

print("Linear SVM Accuracy:")
print(accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

Naive Bayes Accuracy:
0.9835
                 precision    recall  f1-score   support

    electricity       0.98      0.99      0.98       334
     healthcare       0.99      0.98      0.99       333
  ration_aadhar       0.98      0.98      0.98       333
roads_transport       0.99      0.99      0.99       333
     sanitation       0.98      1.00      0.99       333
   water_supply       0.98      0.96      0.97       334

       accuracy                           0.98      2000
      macro avg       0.98      0.98      0.98      2000
   weighted avg       0.98      0.98      0.98      2000

Logistic Regression Accuracy:
0.9955
                 precision    recall  f1-score   support

    electricity       1.00      1.00      1.00       334
     healthcare       1.00      0.99      1.00       333
  ration_aadhar       0.99      1.00      1.00       333
roads_transport       1.00      1.00      1.00       333
     sanitation       0.99      1.00      0.99       333
   water_supply   

# similar for urgency

In [24]:
y_urgency = df["urgency"]

print(y_urgency.value_counts())

urgency
high      3334
low       3334
medium    3332
Name: count, dtype: int64


In [25]:
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X,
    y_urgency,
    test_size=0.2,
    random_state=42,
    stratify=y_urgency
)

print("Training data:", X_train_u.shape)
print("Testing data:", X_test_u.shape)

Training data: (8000, 1798)
Testing data: (2000, 1798)


In [26]:
y_urgency = df["urgency"]

print(y_urgency.value_counts())

urgency
high      3334
low       3334
medium    3332
Name: count, dtype: int64


In [27]:
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X,
    y_urgency,
    test_size=0.2,
    random_state=42,
    stratify=y_urgency
)

print("Training data:", X_train_u.shape)
print("Testing data:", X_test_u.shape)

Training data: (8000, 1798)
Testing data: (2000, 1798)


#  Train Urgency Models

In [28]:
# Naive Bayes
nb_urgency = MultinomialNB()
nb_urgency.fit(X_train_u, y_train_u)
nb_urgency_pred = nb_urgency.predict(X_test_u)


# Logistic Regression
lr_urgency = LogisticRegression(max_iter=1000)
lr_urgency.fit(X_train_u, y_train_u)
lr_urgency_pred = lr_urgency.predict(X_test_u)


# Linear SVM
svm_urgency = LinearSVC()
svm_urgency.fit(X_train_u, y_train_u)
svm_urgency_pred = svm_urgency.predict(X_test_u)

# Evaluation of modells

In [29]:
print("Naive Bayes Accuracy:")
print(accuracy_score(y_test_u, nb_urgency_pred))
print(classification_report(y_test_u, nb_urgency_pred))

print("Logistic Regression Accuracy:")
print(accuracy_score(y_test_u, lr_urgency_pred))
print(classification_report(y_test_u, lr_urgency_pred))

print("Linear SVM Accuracy:")
print(accuracy_score(y_test_u, svm_urgency_pred))
print(classification_report(y_test_u, svm_urgency_pred))

Naive Bayes Accuracy:
0.983
              precision    recall  f1-score   support

        high       0.99      0.98      0.98       667
         low       0.99      0.99      0.99       667
      medium       0.97      0.98      0.98       666

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000

Logistic Regression Accuracy:
0.9965
              precision    recall  f1-score   support

        high       1.00      0.99      0.99       667
         low       1.00      1.00      1.00       667
      medium       0.99      1.00      0.99       666

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Linear SVM Accuracy:
1.0
              precision    recall  f1-score   support

        high       1.00      1.00      1.00       667
         low       1.00      1.00      